# Non Invasive Brain Stimulation (Transtranial Magnetic Stimulation, TMS)

TMS is a non-invasive procedure that uses magnetic fields to stimulate nerve cells in the brain to improve symptoms of neurological or mental health conditions. During a TMS session, an electromagnetic coil is placed against your scalp near your forehead. The electromagnet painlessly delivers a magnetic pulse that stimulates nerve cells in the region of your brain involved in mood control and depression. It's thought that this stimulation may activate brain regions that have decreased activity in depression. TMS is typically used when other treatments for depression, such as medication, hasn't been effective.



In [ ]:
# Install necessary libraries (if not already installed)


!pip install nibabel nilearn ipywidgets matplotlib numpy


In [64]:
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
from nilearn import plotting
from ipywidgets import interact, FloatSlider
from IPython.display import display, clear_output

# --- IMPORTANT: Replace these paths with your actual NIfTI file paths ---
base_image_path = '/content/drive/MyDrive/Workshops_Diplomados/Falan2026_STGO/output/T1.nii.gz'
overlay_image_path = '/content/drive/MyDrive/Workshops_Diplomados/Falan2026_STGO/output/sub-MSC01b_F3_magnE.nii.gz'
#overlay_image_path = '/content/drive/MyDrive/Workshops_Diplomados/Falan2026_STGO/output/sub-MSC01b_F4.nii.gz'



In [65]:

# Load the NIfTI images
try:
    base_img = nib.load(base_image_path)
    overlay_img = nib.load(overlay_image_path)
    print(f"Successfully loaded base image: {base_image_path}")
    print(f"Successfully loaded overlay image: {overlay_image_path}")
except FileNotFoundError:
    print("Error: NIfTI file not found. Please ensure the paths are correct.")
    base_img = None
    overlay_img = None

# --- Calculate ranges for sliders ---

# For Threshold Slider
if overlay_img:
    overlay_data = overlay_img.get_fdata()
    non_nan_values = overlay_data[~np.isnan(overlay_data)]
    if non_nan_values.size > 0:
        overlay_min = np.min(non_nan_values)
        overlay_max = np.max(non_nan_values)
    else:
        overlay_min, overlay_max = 0.0, 1.0 # Default if no valid data
else:
    overlay_min, overlay_max = 0.0, 1.0 # Default if image not loaded

# For Cut_coords Sliders (X, Y, Z)
if base_img:
    affine = base_img.affine
    shape = base_img.shape
    voxel_corners = np.array([
        [0, 0, 0, 1],
        [shape[0]-1, 0, 0, 1],
        [0, shape[1]-1, 0, 1],
        [0, 0, shape[2]-1, 1],
        [shape[0]-1, shape[1]-1, shape[2]-1, 1]
    ])
    world_corners = nib.affines.apply_affine(affine, voxel_corners[:, :3])
    x_min, y_min, z_min = np.min(world_corners, axis=0)
    x_max, y_max, z_max = np.max(world_corners, axis=0)
    default_cut_coords_voxel = (np.array(shape) - 1) / 2.0
    default_cut_coords = nib.affines.apply_affine(affine, default_cut_coords_voxel)
    x_default, y_default, z_default = default_cut_coords
else:
    x_min, x_max, x_default = -50, 50, 0
    y_min, y_max, y_default = -50, 50, 0
    z_min, z_max, z_default = -50, 50, 0

# --- Interactive Plotting Function ---
if base_img and overlay_img:
    def plot_interactive_stat_map_with_coords(threshold_value, x_coord, y_coord, z_coord):
        # Removed with clear_output(wait=True): as it can cause issues in this context
        fig = plt.figure(figsize=(10, 8))
        plotting.plot_stat_map(
            stat_map_img=overlay_img,
            bg_img=base_img,
            display_mode='ortho',
            cut_coords=(x_coord, y_coord, z_coord),
            title=f'Interactive Overlay (Thresh > {threshold_value:.2f}, Coords: ({x_coord:.0f},{y_coord:.0f},{z_coord:.0f}))',
            threshold=threshold_value,
            cmap='cold_hot',
            colorbar=True,
            figure=fig
        )
        plt.show()
        plt.close(fig) # Close the figure to prevent memory issues and display only the current plot

    # Create sliders
    threshold_slider = FloatSlider(
        min=overlay_min,
        max=overlay_max,
        step=(overlay_max - overlay_min) / 100 if (overlay_max - overlay_min) > 0 else 0.1,
        value=overlay_min + (overlay_max - overlay_min) * 0.5,
        description='Threshold:',
        continuous_update=True
    )
    x_slider = FloatSlider(
        min=x_min, max=x_max, step=1.0, value=x_default,
        description='X-coord:', continuous_update=True
    )
    y_slider = FloatSlider(
        min=y_min, max=y_max, step=1.0, value=y_default,
        description='Y-coord:', continuous_update=True
    )
    z_slider = FloatSlider(
        min=z_min, max=z_max, step=1.0, value=z_default,
        description='Z-coord:', continuous_update=True
    )

    # Link sliders to the plotting function
    interact(
        plot_interactive_stat_map_with_coords,
        threshold_value=threshold_slider,
        x_coord=x_slider,
        y_coord=y_slider,
        z_coord=z_slider
    )
else:
    print("Cannot create interactive plot: Images were not loaded successfully. Please check file paths.")

Successfully loaded base image: /content/drive/MyDrive/Workshops_Diplomados/Falan2026_STGO/output/T1.nii.gz
Successfully loaded overlay image: /content/drive/MyDrive/Workshops_Diplomados/Falan2026_STGO/output/sub-MSC01b_F3_magnE.nii.gz


interactive(children=(FloatSlider(value=0.7834325432777405, description='Threshold:', max=1.566865086555481, s…